### Creates and maintains `openalex.authors.parsed_names_normalized` table

Self-contained notebook that gathers new author names, parses them, and normalizes in one pass.

Adds on top of basic parsing:
- **Romanization**: `unidecode` for CJK/Cyrillic/Arabic → ASCII
- **ALL CAPS surname detection**: fixes French-style "DUPONT Jean" parsing
- **Middle initials detection**: periods, caps, or short strings → initials
- **Always-present initial columns**: `first_initial`, `middle_initials`

In [ ]:
%pip install nameparser unidecode

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.authors.parsed_names_normalized (
  raw_author_name STRING,
  parsed_name STRUCT<
      title: STRING,
      first: STRING,
      middle: STRING,
      last: STRING,
      suffix: STRING,
      nickname: STRING
  >,
  created_datetime TIMESTAMP,
  normalized_first STRING,
  first_initial STRING,
  normalized_middle STRING,
  middle_initials STRING,
  middle_initial_count INT,
  normalized_last STRING
)
USING DELTA
CLUSTER BY (raw_author_name)

#### Imports, constants, CJK support

In [ ]:
import unicodedata
import re

from unidecode import unidecode
from nameparser import HumanName

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


# -- Schema for parsed_name struct --

parsed_name_schema = StructType([
    StructField('title', StringType(), True),
    StructField('first', StringType(), True),
    StructField('middle', StringType(), True),
    StructField('last', StringType(), True),
    StructField('suffix', StringType(), True),
    StructField('nickname', StringType(), True)
])

# -- CJK support --

# Common two-character Chinese surnames (复姓)
COMPOUND_SURNAMES = {
    '欧阳', '太史', '端木', '上官', '司马', '东方', '独孤', '南宫', '万俟', 
    '闻人', '夏侯', '诸葛', '尉迟', '公羊', '赫连', '澹台', '皇甫', '宗政',
    '濮阳', '公冶', '太叔', '申屠', '公孙', '慕容', '仲孙', '钟离', '长孙',
    '宇文', '司徒', '鲜于', '司空', '闾丘', '子车', '亓官', '司寇', '巫马',
    '公西', '颛孙', '壤驷', '公良', '漆雕', '乐正', '宰父', '谷梁', '拓跋',
    '夹谷', '轩辕', '令狐', '段干', '百里', '呼延', '东郭', '南门', '羊舌',
    '微生', '公户', '公玉', '公仪', '梁丘', '公仲', '公上', '公门', '公山',
    '公坚', '左丘', '公伯', '西门', '公祖', '第五', '公乘', '贯丘', '公皙',
    '南荣', '东里', '东宫', '仲长', '子书', '子桑', '即墨', '达奚', '褚师',
    # Traditional variants
    '歐陽', '司馬', '東方', '獨孤', '南宮', '諸葛', '尉遲', '赫連', '澹臺',
    '皇甫', '濮陽', '慕容', '鍾離', '長孫', '宇文', '鮮于', '閭丘', '顓孫',
    '漆雕', '樂正', '穀梁', '拓跋', '夾谷', '軒轅', '段幹', '東郭', '南門',
    '羊舌', '梁丘', '左丘', '西門', '東里', '東宮', '仲長'
}

def is_cjk(char):
    """Check if a character is CJK (Chinese/Japanese/Korean)."""
    cp = ord(char)
    return (
        (0x4E00 <= cp <= 0x9FFF) or    # CJK Unified Ideographs
        (0x3400 <= cp <= 0x4DBF) or    # CJK Unified Ideographs Extension A
        (0x20000 <= cp <= 0x2A6DF) or  # CJK Unified Ideographs Extension B
        (0xF900 <= cp <= 0xFAFF) or    # CJK Compatibility Ideographs
        (0x2F800 <= cp <= 0x2FA1F)     # CJK Compatibility Ideographs Supplement
    )

def is_all_cjk(s):
    """Check if string is entirely CJK characters (ignoring whitespace)."""
    chars = s.replace(' ', '')
    return len(chars) > 0 and all(is_cjk(c) for c in chars)

def split_chinese_name(name):
    """
    Split a Chinese name written without spaces into (surname, given_name).
    Assumes Eastern order (surname first).
    """
    # Check for compound surname first
    if len(name) >= 2 and name[:2] in COMPOUND_SURNAMES:
        return name[:2], name[2:]
    # Default: single-character surname
    return name[0], name[1:]

def parse_name(name_string):
    """
    Parse a name, with special handling for unsplit Chinese names.
    Returns a HumanName object with corrected first/last for Chinese names.
    """
    name_string = name_string.strip()
    
    # Only special case: All CJK with no spaces - assume Eastern order, split it
    if is_all_cjk(name_string) and ' ' not in name_string:
        surname, given = split_chinese_name(name_string)
        result = HumanName()
        result.last = surname
        result.first = given
        return result
    
    # Everything else: use standard nameparser (assumes Western order)
    return HumanName(name_string)


def clean_name_component(s: str) -> str:
    """Remove diacritics, lowercase, and remove periods."""
    if s is None or s == '':
        return ''
    # Remove diacritics (normalize to NFD, remove combining characters)
    normalized = unicodedata.normalize('NFD', s)
    without_diacritics = ''.join(c for c in normalized if unicodedata.category(c) != 'Mn')
    # Lowercase and remove periods
    cleaned = without_diacritics.lower().replace('.', '')
    # Collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


def parse_human_name(name: str) -> dict:
    """
    Parse a name using nameparser.HumanName and return cleaned components.
    Includes special handling for CJK names without spaces.
    Returns dict with: title, first, middle, last, suffix, nickname
    All values are cleaned: diacritics removed, lowercased, periods removed.
    """
    if name is None or not isinstance(name, str) or name.strip() == '':
        return {
            'title': '',
            'first': '',
            'middle': '',
            'last': '',
            'suffix': '',
            'nickname': ''
        }
    
    try:
        parsed = parse_name(name)  # use our CJK-aware parser
        return {
            'title': clean_name_component(parsed.title),
            'first': clean_name_component(parsed.first),
            'middle': clean_name_component(parsed.middle),
            'last': clean_name_component(parsed.last),
            'suffix': clean_name_component(parsed.suffix),
            'nickname': clean_name_component(parsed.nickname)
        }
    except Exception:
        return {
            'title': '',
            'first': '',
            'middle': '',
            'last': '',
            'suffix': '',
            'nickname': ''
        }


# Top two-letter first names from name_freq_first_unified (freq > 0, ASCII, has vowel).
# Used to prevent misclassifying real names like "Li" or "Bo" as initials.
TWO_LETTER_NAMES = {
    'li', 'bo', 'yi', 'yu', 'ko', 'so', 'qi', 'yo', 'ma', 'xu', 'na', 'lu',
    'ke', 'ha', 'mi', 'ye', 'su', 'ei', 'ta', 'xi', 'he', 'ao', 'ia', 'az',
    'di', 'ji', 'hu', 'fu', 'jo', 'ka', 'do', 'en', 'wu', 'ed', 'ya', 'ki',
    'ak', 'oh', 'vo', 'ag', 'ur', 'ej', 'mu', 'le', 'ge', 'al', 'se', 'ah',
    'zo', 'vi', 'um', 'oo', 'ri', 'ga', 'ja', 'je', 'at', 'ju', 'da', 'an',
    'ca', 'eh', 'vu', 'ug', 'ea', 'or', 'pa', 'ay', 'is', 'ne', 'av', 'fe',
    'co', 'et', 'ru', 'aj', 'ze', 'de', 'mo', 'ni', 'si', 'pu', 'fa', 'za',
    'el', 'xo', 'ku', 'ut', 'wa', 'eg', 'uy', 'iu', 'ie', 'pe', 'oz', 'ae',
    'ho', 'cu', 'ax', 'nu', 'ez', 'ai', 'em', 'om', 'ce', 'tu', 'zi', 'ro',
    'bi', 'ad', 'po', 'ou', 'ui', 'ua', 'io', 'on', 'go', 'og', 'ar', 'ba',
    'gu', 'te', 'ra', 'la', 'ac', 'er', 'bu', 'ti', 'gi', 'du', 'ci', 'es',
    'ev', 'pi', 'va', 'zu', 'id', 've', 'oi', 'ol', 'be', 'ot', 'ec', 'ef',
    'ud', 'iv', 'ii', 'wo', 'wi', 'ee', 'iz', 'od', 'xa', 'no', 'oa', 'me',
    'uz', 'ul', 'oy', 'uu', 'xe', 'as', 'il', 'us', 'un', 'ok', 're',
}

#### Normalization functions

In [ ]:
MAX_MIDDLE_INITIALS = 3  # More than 3 "initials" is certainly a full name


def normalize_name_part(s):
    """Romanize and clean a name part to ASCII lowercase.
    
    Pipeline: unidecode → NFD strip diacritics → lowercase → drop periods
    → keep only letters/spaces/hyphens → collapse whitespace.
    """
    if not s or not isinstance(s, str):
        return ''
    s = s.strip()
    if not s:
        return ''
    # Romanize non-Latin scripts (CJK, Cyrillic, Arabic, etc.)
    result = unidecode(s)
    # Strip remaining diacritics via NFD decomposition
    result = unicodedata.normalize('NFD', result)
    result = ''.join(c for c in result if unicodedata.category(c) != 'Mn')
    # Lowercase and remove periods
    result = result.lower().replace('.', '')
    # Keep only letters, spaces, and hyphens
    result = re.sub(r'[^a-z\s-]', '', result)
    # Collapse whitespace
    result = re.sub(r'\s+', ' ', result).strip()
    return result


def detect_all_caps_surname(raw_name):
    """Detect French-style ALL CAPS surname in a raw author name.
    
    Rules (ALL must be true):
    - Exactly one token is ALL CAPS
    - That token is purely alphabetic (no periods, digits, hyphens)
    - That token has length >= 3
    - That token contains at least one vowel (AEIOU)
    
    Returns (first_raw, middle_raw, last_raw) if detected, else None.
    The ALL CAPS token becomes last; remaining tokens become first + middle.
    """
    if not raw_name or not isinstance(raw_name, str):
        return None
    tokens = raw_name.split()
    if len(tokens) < 2:
        return None

    caps_indices = [
        i for i, t in enumerate(tokens)
        if t.isalpha() and t.isupper() and len(t) >= 3 and re.search(r'[AEIOU]', t)
    ]
    if len(caps_indices) != 1:
        return None

    caps_idx = caps_indices[0]
    surname = tokens[caps_idx]
    remaining = [t for i, t in enumerate(tokens) if i != caps_idx]

    if len(remaining) == 1:
        return (remaining[0], '', surname)
    else:
        return (remaining[0], ' '.join(remaining[1:]), surname)


# Pre-compiled regex for finding period-separated initial groups like "H.W." or "J.R.R."
_RE_PERIOD_GROUPS = re.compile(r'(?<!\w)([A-Za-z]\.(?:\s*[A-Za-z]\.)*)') 


def is_middle_initials(norm_middle, raw_author_name):
    """Determine if a parsed middle name represents initials.
    
    Three rules (any triggers initials):
    1. Period-separated initials in raw name that MATCH the normalized middle
       (e.g. raw "H.W." matches norm_middle "hw"). Prevents false positives
       from period patterns elsewhere in the name (e.g. first-name initials).
    2. Middle tokens are ALL CAPS in raw name, but whole name isn't all caps.
       Skipped for comma-separated names ("Last, First Middle") where token
       positions don't correspond to first/middle/last.
    3. Length < 3 (after stripping spaces), unless in TWO_LETTER_NAMES
    
    Args:
        norm_middle: normalized middle name (lowercase, ASCII)
        raw_author_name: original raw string with casing intact.
            When CAPS correction fires, caller passes the reconstructed name
            (surname removed) so rule 2 checks the right tokens.
    """
    if not norm_middle:
        return False

    m_stripped = re.sub(r'\s', '', norm_middle)
    if not m_stripped:
        return False

    # Hard cap: more than MAX_MIDDLE_INITIALS chars is never initials
    if len(m_stripped) > MAX_MIDDLE_INITIALS:
        return False

    raw = raw_author_name if isinstance(raw_author_name, str) else ''

    # Rule 1: Period-separated initials in raw name that match our middle
    # E.g. "H.W." → cleaned "hw" must equal m_stripped "hw"
    if raw:
        for match in _RE_PERIOD_GROUPS.finditer(raw):
            cleaned = re.sub(r'[\s.]', '', match.group()).lower()
            if cleaned == m_stripped:
                return True

    # Rule 2: Middle tokens in raw name are ALL CAPS, but whole name isn't.
    # Skip for comma-separated names — token positions are unreliable.
    if raw and ',' not in raw and not raw.isupper():
        tokens = raw.split()
        if len(tokens) >= 3:
            middle_tokens = tokens[1:-1]
            if middle_tokens and all(t.isupper() for t in middle_tokens):
                return True

    # Rule 3: Short middle (< 3 chars), unless it's a known two-letter name
    if len(m_stripped) < 3:
        if len(m_stripped) == 2 and m_stripped in TWO_LETTER_NAMES:
            return False  # real name, not initials
        return True  # single char or unknown 2-char → initials

    return False


def process_row(raw_name, parsed_first, parsed_middle, parsed_last):
    """Process one row: normalize names, detect initials, correct ALL CAPS surnames.
    
    Names and initials are mutually exclusive:
    - If we only have an initial, the name is NULL (unknown).
    - If we have a full name, the initial is derived from it.
    - first_initial and middle_initials are always populated when any info exists.
    
    Returns dict with: normalized_first, first_initial, normalized_middle,
    middle_initials, middle_initial_count, normalized_last.
    """
    # Sanitize inputs — pandas NaN is truthy but not a string
    raw_name = raw_name if isinstance(raw_name, str) else ''
    parsed_first = parsed_first if isinstance(parsed_first, str) else ''
    parsed_middle = parsed_middle if isinstance(parsed_middle, str) else ''
    parsed_last = parsed_last if isinstance(parsed_last, str) else ''

    # Step 1: Check for ALL CAPS surname in raw name
    caps_result = detect_all_caps_surname(raw_name)
    if caps_result:
        first_src, middle_src, last_src = caps_result
        # Reconstruct raw name without the CAPS surname so is_middle_initials
        # checks casing on the remaining tokens, not the removed surname.
        parts = [first_src] + ([middle_src] if middle_src else [])
        raw_for_initials = ' '.join(parts)
    else:
        first_src = parsed_first
        middle_src = parsed_middle
        last_src = parsed_last
        raw_for_initials = raw_name

    # Step 2: Normalize all parts (unidecode + clean)
    norm_first = normalize_name_part(first_src)
    norm_middle = normalize_name_part(middle_src)
    norm_last = normalize_name_part(last_src)

    # Step 3: First name vs initial (mutually exclusive)
    # If normalized first is a single char, it's an initial — we don't know the name.
    if not norm_first:
        first_name = None
        first_initial = None
    elif len(norm_first) == 1:
        first_name = None           # unknown — we only have an initial
        first_initial = norm_first   # "j"
    else:
        first_name = norm_first      # "jason"
        first_initial = norm_first[0]  # "j"

    # Step 4: Middle name vs initials (mutually exclusive)
    if not norm_middle:
        middle_name = None
        mid_initials = None
        mid_count = 0
    elif is_middle_initials(norm_middle, raw_for_initials):
        middle_name = None  # unknown — we only have initials
        mid_initials = re.sub(r'[^a-z]', '', norm_middle)
        mid_count = len(mid_initials) if mid_initials else 0
    else:
        middle_name = norm_middle        # "william"
        mid_initials = norm_middle[0]    # "w"
        mid_count = 1

    return {
        'normalized_first': first_name,
        'first_initial': first_initial,
        'normalized_middle': middle_name,
        'middle_initials': mid_initials,
        'middle_initial_count': mid_count,
        'normalized_last': norm_last or None,
    }

#### Pandas UDFs

In [ ]:
@F.pandas_udf(parsed_name_schema)
def parse_names_batch(names: pd.Series) -> pd.DataFrame:
    """Vectorized UDF to parse a batch of names."""
    results = [parse_human_name(n) for n in names]
    return pd.DataFrame(results)


norm_schema = StructType([
    StructField('normalized_first', StringType(), True),
    StructField('first_initial', StringType(), True),
    StructField('normalized_middle', StringType(), True),
    StructField('middle_initials', StringType(), True),
    StructField('middle_initial_count', IntegerType(), True),
    StructField('normalized_last', StringType(), True),
])

_NORM_COLUMNS = [f.name for f in norm_schema.fields]


@F.pandas_udf(norm_schema)
def normalize_names_batch(
    raw_names: pd.Series,
    firsts: pd.Series,
    middles: pd.Series,
    lasts: pd.Series,
) -> pd.DataFrame:
    """Vectorized UDF: normalize and classify name parts."""
    results = [
        process_row(rn, f, m, l)
        for rn, f, m, l in zip(raw_names, firsts, middles, lasts)
    ]
    return pd.DataFrame(results, columns=_NORM_COLUMNS)

#### Get new names to process

In [ ]:
new_names_df = spark.sql(f"""
    WITH distinct_names AS (
        SELECT DISTINCT TRIM(author.name) AS raw_author_name
        FROM openalex.works.locations_mapped
        LATERAL VIEW explode(authors) AS author
        WHERE author.name IS NOT NULL 
          AND TRIM(author.name) != ''
        
        UNION
        
        SELECT DISTINCT TRIM(full_name) AS raw_author_name
        FROM openalex.authors.openalex_authors
        WHERE full_name IS NOT NULL 
          AND TRIM(full_name) != ''

        UNION

        SELECT DISTINCT TRIM(raw_author_name) AS raw_author_name
        FROM openalex.works_legacy.work_authors
        WHERE raw_author_name IS NOT NULL
          AND TRIM(raw_author_name) != ''
    )
    SELECT dn.raw_author_name
    FROM distinct_names dn
    LEFT ANTI JOIN openalex.authors.parsed_names_normalized existing
        ON dn.raw_author_name = existing.raw_author_name
""")

new_count = new_names_df.count()
print(f"Found {new_count:,} new distinct author names to parse and normalize")

#### Run it

In [ ]:
target_table = "openalex.authors.parsed_names_normalized"

if new_count > 0:
    # Repartition for parallelism
    new_names_df = new_names_df.repartition(100)

    # Step 1: Parse names
    parsed_df = new_names_df.withColumn(
        "parsed_name", parse_names_batch(F.col("raw_author_name"))
    )

    # Step 2: Normalize parsed names
    result_df = parsed_df.withColumn(
        "norm",
        normalize_names_batch(
            F.col("raw_author_name"),
            F.col("parsed_name.first"),
            F.col("parsed_name.middle"),
            F.col("parsed_name.last"),
        ),
    ).select(
        "raw_author_name",
        "parsed_name",
        F.current_timestamp().alias("created_datetime"),
        F.col("norm.normalized_first"),
        F.col("norm.first_initial"),
        F.col("norm.normalized_middle"),
        F.col("norm.middle_initials"),
        F.col("norm.middle_initial_count"),
        F.col("norm.normalized_last"),
    )

    result_df.write.format("delta").mode("append").saveAsTable(target_table)

    print(f"Added {new_count:,} parsed + normalized names to {target_table}")
else:
    print("No new names to parse")

#### Verification

In [ ]:
target_table = "openalex.authors.parsed_names_normalized"
total = spark.table(target_table).count()
print(f"Total rows in {target_table}: {total:,}")

# Test 1: Non-ASCII eliminated
non_ascii = spark.sql(f"""
    SELECT COUNT(*) AS non_ascii_count FROM {target_table}
    WHERE normalized_last RLIKE '[^\\x00-\\x7F]'
       OR normalized_first RLIKE '[^\\x00-\\x7F]'
       OR normalized_middle RLIKE '[^\\x00-\\x7F]'
""").collect()[0]['non_ascii_count']
print(f"Test 1 - Non-ASCII in normalized columns: {non_ascii} (should be 0)")

In [ ]:
# Test 2: Names and initials are mutually exclusive
# If we have a name, we also have an initial. If we only have an initial, name is NULL.
name_without_initial = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {target_table}
    WHERE normalized_first IS NOT NULL AND first_initial IS NULL
""").collect()[0]['c']
initial_with_name = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {target_table}
    WHERE normalized_first IS NULL AND first_initial IS NOT NULL AND LENGTH(first_initial) = 1
       AND normalized_last IS NOT NULL
""").collect()[0]['c']
mid_name_with_mid_initials_equal = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {target_table}
    WHERE normalized_middle IS NOT NULL AND normalized_middle = middle_initials
""").collect()[0]['c']
print(f"Test 2a - Name without initial: {name_without_initial} (should be 0)")
print(f"Test 2b - Initial-only first names (name is NULL): {initial_with_name:,} (should be > 0)")
print(f"Test 2c - Middle name == middle initials (should be 0 — mutually exclusive): {mid_name_with_mid_initials_equal}")

In [ ]:
# Test 3: Spot-check known cases
spot_checks = spark.sql(f"""
    SELECT raw_author_name, normalized_first, first_initial,
           normalized_middle, middle_initials, middle_initial_count, normalized_last
    FROM {target_table}
    WHERE raw_author_name IN (
        'George H.W. Bush', 'J.R.R. Tolkien', 'C.S. Lewis', 'John William Smith'
    )
""")
spot_checks.show(truncate=False)

In [ ]:
# Test 4: ALL CAPS surname detection
caps_checks = spark.sql(f"""
    SELECT raw_author_name, normalized_first, normalized_last
    FROM {target_table}
    WHERE raw_author_name IN ('DUPONT Jean', 'Marie CURIE', 'AB Smith', 'JONES SMITH')
""")
caps_checks.show(truncate=False)

In [ ]:
# Summary stats
spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN normalized_first IS NOT NULL THEN 1 ELSE 0 END) AS has_first_name,
        SUM(CASE WHEN normalized_first IS NULL AND first_initial IS NOT NULL THEN 1 ELSE 0 END) AS first_initial_only,
        SUM(CASE WHEN normalized_middle IS NOT NULL THEN 1 ELSE 0 END) AS has_middle_name,
        SUM(CASE WHEN normalized_middle IS NULL AND middle_initials IS NOT NULL THEN 1 ELSE 0 END) AS middle_initial_only,
        SUM(CASE WHEN middle_initials IS NULL THEN 1 ELSE 0 END) AS no_middle_at_all,
        MAX(middle_initial_count) AS max_middle_initial_count
    FROM {target_table}
""").show(truncate=False)